In [ ]:
import os
from pathlib import Path
import pandas as pd
import time
from datetime import datetime

import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from src.analyse.utils import get_straight_line_distance

load_dotenv()

In [ ]:
calculate_distance = False
save_figs = False

In [ ]:
today = pd.Timestamp.now(tz="UTC")
today

In [ ]:
output_dir = Path(f"outputs/nb03_analyse_listings_{datetime.now().strftime('%Y%m%d-%H%M%S')}")
if save_figs:
    os.makedirs(output_dir, exist_ok=True)

In [ ]:
input_dir = Path('outputs/funda/')
files = sorted(list(input_dir.glob('*.csv')))
files.remove(input_dir / 'extra_info.csv')
files

In [ ]:
df_extra = pd.read_csv(input_dir / 'extra_info.csv')
df_extra.head()

In [ ]:
ind = -1
df_raw = pd.read_csv(files[ind])
df_raw.head()

In [ ]:
df_raw.dtypes

In [ ]:
df = df_raw.copy()
df["publish_date"] = pd.to_datetime(df_raw["publish_date"], format="mixed", utc=True)
df["year_quarter"] = df["publish_date"].dt.to_period("Q")
df["year_month"] = df["publish_date"].dt.to_period("M")
df["price_per_sqm"] = df["price"] / df["floor_area"]
df["status"] = df["status"].apply(lambda x: x if x not in ["none"] else "available")
df["time_on_market_days"] = (today - df["publish_date"]).dt.days
df["time_on_market_days"] = df["time_on_market_days"].where(df["status"] == "available", None)
df["time_on_market_months"] = df["time_on_market_days"] / 30.44
df.sort_values("publish_date", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

In [ ]:
df = pd.merge(df, df_extra, on="address", how="left")
df.head()

In [ ]:
df["extra_score"] = df[df_extra.columns.drop("address")].sum(axis=1)
df["workload_score"] = df[["floor_done", "bathroom_renovated", "kitchen_renovated"]].sum(axis=1)
df.tail()

In [ ]:
df_dists = None

In [ ]:
if calculate_distance:
    reference_address = os.getenv("REFERENCE_ADDRESS", "Amsterdam Central Station")
    print(f"Reference address: {reference_address}")

    distances = []
    for ind, row in df.iterrows():
        print(f"row {ind}", end="\r")
        address = row["address"]
        dist = get_straight_line_distance(address, reference_address)
        dist["id"] = row["id"]
        distances.append(dist)

        time.sleep(0.2)

    df_dists = pd.DataFrame(distances)
    display(df_dists.head())

    df = pd.merge(df, df_dists, on="id")
    df.head()


In [ ]:
if df_dists is not None and save_figs:
    df_dists.to_csv(output_dir / f"distances_from_{reference_address.replace(' ', '_')}.csv")
    print("Saved dists dataframe")

In [ ]:
df_grouped = df.groupby("year_month").agg(
    num_listings=("id", "count"),
    price_avg=("price", "mean"),
    price_median=("price", "median"),
    price_per_sqm=("price_per_sqm", "mean"),
    living_area=("floor_area", "mean"),
)
df_grouped.reset_index(inplace=True)
df_grouped["year_month"] = df_grouped["year_month"].dt.to_timestamp()
df_grouped.sort_values("year_month", inplace=True)
df_grouped.dtypes

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.violinplot(
    data=df,
    x="year_quarter",
    y="price_per_sqm",
    inner="quartile",
)

plt.axhline(
    y=df["price_per_sqm"].mean(),
    xmin=-0.5,
    xmax=len(df_grouped) - 0.5,
    color="red",
    linestyle="dashed",
    label="Overall Average",
)

plt.xticks(rotation=45, ha="right")
plt.legend()
plt.title("Price per sqm by Year Quarter")
plt.xlabel("Year Quarter")
plt.ylabel("Price per sqm")

if save_figs:
    plt.savefig(output_dir / "price_per_sqm_violinplot.png", bbox_inches="tight")

plt.show()

In [ ]:
def plot_price_per_sqm_over_time(df):
    _ = plt.figure(figsize=(12, 6))

    sns.boxplot(
        data=df,
        x="year_month",
        y="price_per_sqm",
    )

    avg = df["price_per_sqm"].mean()
    plt.axhline(
        y=avg,
        color="red",
        linestyle="dashed",
        label=f"Overall Average ({avg:,.0f} €/m²)",
    )

    plt.title("Average Price per Square Meter Over Time")
    plt.xticks(rotation=45, ha="right")
    plt.legend()

    if save_figs:
        plt.savefig(output_dir / "price_per_sqm_over_time.png", bbox_inches="tight")

    plt.show()

In [ ]:
plot_price_per_sqm_over_time(df)

In [ ]:
df.head()

In [ ]:
df.status.unique()

In [ ]:
def filter_df(df, match_criteria):
    df_filtered = df.copy()
    for col, criteria in match_criteria.items():
        if isinstance(criteria, tuple) and len(criteria) == 2 and all(isinstance(c, (int, float)) for c in criteria):
            df_filtered = df_filtered[(df_filtered[col] >= criteria[0]) & (df_filtered[col] <= criteria[1])]
        elif isinstance(criteria, tuple):
            df_filtered = df_filtered[df_filtered[col].isin(criteria)]
    return df_filtered

In [ ]:
match_criteria = {
    "floor_area": (50, 100),
    "number_of_bedrooms": (1, 4),
    # "energy_label": ("D", "C", "B", "A", "A+", "A++"),
    "status": ("available", "under_bid", "sold_under_reservation", "sold"),
    # "status": ("available", "under_bid"),
}

df_filtered = filter_df(df, match_criteria)
df_filtered.head()

In [ ]:
plot_price_per_sqm_over_time(df_filtered)

In [ ]:
cols_of_interest = [
    "address", "price_per_sqm", "price", "floor_area", "number_of_bedrooms", "distance_km", "estimated_walking_min", "energy_label", "status", "url", "publish_date", "time_on_market_months", "extra_score", "workload_score"
]
cols_not_available = [col for col in cols_of_interest if col not in df_filtered.columns]
for col in cols_not_available:
    cols_of_interest.remove(col)

df_filtered[cols_of_interest]

In [ ]:
df_filtered[cols_of_interest].sort_values("price_per_sqm")

In [ ]:
criteria_available = {
    "status": ("available", "under_bid"),
}
df_available = filter_df(df_filtered, criteria_available)
df_available

In [ ]:
criteria = {
    "status": ("available", "under_bid"),
    "price": (0, 400000),
}
df_available = filter_df(df, criteria)
if save_figs:
    df_available.to_csv(output_dir / "df_available.csv", index=False)
df_available

In [ ]:
sns.scatterplot(
    data=df_available,
    x="floor_area",
    y="price_per_sqm",
    size="extra_score",
    hue="workload_score",
)

plt.grid()
plt.title("Price per Square Meter vs. Floor Area")
if save_figs:
    plt.savefig(output_dir / "price_per_sqm_vs_floor_area.png", bbox_inches="tight")
plt.show()

In [ ]:
_ = plt.figure(figsize=(10,10))
sns.scatterplot(
    data=df,
    x="floor_area",
    y="price_per_sqm",
    hue="year_month",
    palette="viridis",
)

plt.grid()
plt.title("Price per sqm by Floor Area and Year Month")
plt.xlabel("Floor Area (m²)")
plt.ylabel("Price per sqm (€)")

if save_figs:
    plt.savefig(output_dir / "price_per_sqm_by_floor_area_and_year_month.png", bbox_inches="tight")